# Instalación de librerías

In [1]:
!pip install -q scrapy
!pip install -q newspaper3k
!pip install -q lxml_html_clean

# Extraer enlaces de noticias

Eliminar csv para generar uno nuevo

In [ ]:
# # Eliminar csv
# import os
# output_file_csv = "news_output.csv"
# if os.path.exists(output_file_csv):
#     os.remove(output_file_csv)

Ejecutar spider de scrapy para generar el archivo csv con los enlaces

In [ ]:
import os
from datetime import date

hoy = date.today().strftime("%d-%m-%Y")
comando = f"scrapy crawl news_extractor -o news_scraper/csv/{hoy}.csv -s LOG_ENABLED=False -s ROBOTSTXT_OBEY=False"
os.system(comando)


In [8]:
!scrapy crawl news_extractor -o news_scraper/csv/hoy.csv -s LOG_ENABLED=False -s ROBOTSTXT_OBEY=False

Fecha de inicio: 2025-03-28
Robots URL: https://www.elcomercio.es/robots.txt
Robots URL: https://www.europapress.es/robots.txt
Robots URL: https://www.20minutos.es/robots.txt
Robots URL: https://www.lne.es/robots.txt
Robots URL: https://www.elespanol.com/robots.txt
Robots URL: https://www.eldiario.es/robots.txt
Robots URL: https://www.elconfidencial.com/robots.txt
Robots URL: https://www.rtpa.es/robots.txt
Robots URL: https://www.larazon.es/robots.txt
Robots URL: https://migijon.com/robots.txt
Se encontraron los siguientes enlaces xml: {'https://www.elcomercio.es/sitemap.incremental.xml', 'https://www.elcomercio.es/sitemap.xml'}
Accediendo al siguiente enlace... https://www.elcomercio.es/sitemap.incremental.xml
Accediendo al siguiente enlace... https://www.elcomercio.es/sitemap.xml
Robots URL: https://www.lavanguardia.com/robots.txt
Robots URL: https://www.culturalgijonesa.org/robots.txt
Robots URL: https://www.nortes.me/robots.txt
Robots URL: https://www.elperiodico.com/robots.txt
Rob

Fecha de inicio: 2025-03-28
Robots URL: https://www.20minutos.es/robots.txt
Robots URL: https://www.elcomercio.es/robots.txt
Robots URL: https://www.europapress.es/robots.txt
Robots URL: https://www.culturalgijonesa.org/robots.txt
Robots URL: https://www.lne.es/robots.txt
Robots URL: https://www.eldiario.es/robots.txt
Robots URL: https://www.elespanol.com/robots.txt
Robots URL: https://www.larazon.es/robots.txt
Robots URL: https://www.rtpa.es/robots.txt
Robots URL: https://www.elconfidencial.com/robots.txt
Se encontraron los siguientes enlaces xml: {'https://www.elcomercio.es/sitemap.incremental.xml', 'https://www.elcomercio.es/sitemap.xml'}
Accediendo al siguiente enlace... https://www.elcomercio.es/sitemap.incremental.xml
Accediendo al siguiente enlace... https://www.elcomercio.es/sitemap.xml
Robots URL: https://migijon.com/robots.txt
Robots URL: https://www.nortes.me/robots.txt
Robots URL: https://www.tribunasalamanca.com/robots.txt
Robots URL: https://www.lavanguardia.com/robots.tx

# Cargar datos obtenidos

In [ ]:
import pandas as pd
from datetime import datetime
import numpy as np

# Cargar la data extraída
data_df = pd.read_csv("news_output.csv")

# Convertir a tipo datetime usando UTC para tener un formato estandarizado
data_df['fecha_publicacion'] = pd.to_datetime(data_df['fecha_publicacion'], format='mixed', errors='coerce', utc=True)
# Convertir a zona horaria de españa
data_df['fecha_publicacion'] = data_df['fecha_publicacion'].dt.tz_convert('Europe/Madrid')
# Eliminar duplicados y ordenar por fecha
data_df = data_df.drop_duplicates().sort_values(by='fecha_publicacion', ascending=False).reset_index(drop=True)
data_df

,fuente,url,titulo,fecha_publicacion
0,Levante-EMV,https://www.levante-emv.com/economia/2025/03/1...,Digitaliza tu empresa con éxito: descubre las ...,2025-03-17 07:30:00+01:00
1,MENORCA,https://www.menorca.info/hemeroteca.html,NaN,2025-03-17 05:51:28+01:00
2,MENORCA,https://www.menorca.info/opinion.html,NaN,2025-03-17 05:51:28+01:00
3,MENORCA,https://www.menorca.info/quienes_somos.html,NaN,2025-03-17 05:51:28+01:00
4,MENORCA,https://www.menorca.info/contacto/publicidad.html,NaN,2025-03-17 05:51:28+01:00
...,...,...,...,...
27517,Santander Digital 24 Horas,https://www.santanderdigital24horas.com/redotp...,NaN,NaT
27518,Santander Digital 24 Horas,https://www.santanderdigital24horas.com/lexiwa...,NaN,NaT
27519,Santander Digital 24 Horas,https://www.santanderdigital24horas.com/pablo-...,NaN,NaT
27520,Santander Digital 24 Horas,https://www.santanderdigital24horas.com/medida...,NaN,NaT


In [ ]:
data_df['fuente'].value_counts()

,count
fuente,
La Voz de Galicia,1650
La Vanguardia,1339
Gente Digital,1335
ABC,1210
Europa Press,1166
...,...
Diario del Puerto,5
Deportes Menorca,5
Tomelloso,3


## Eliminar enlaces duplicados

In [ ]:
def remover_url_duplicados(df):
    df['non_null_count'] = df.notnull().sum(axis=1)
    df = df.sort_values(by=['url', 'non_null_count'], ascending=[True, False])
    df = df.drop_duplicates(subset=['url'], keep='first')
    df = df.drop(columns=['non_null_count'])
    df = df.sort_values(by='fecha_publicacion', ascending=False).reset_index(drop=True)
    return df

data_df = remover_url_duplicados(data_df)
data_df

,fuente,url,titulo,fecha_publicacion
0,Levante-EMV,https://www.levante-emv.com/economia/2025/03/1...,Digitaliza tu empresa con éxito: descubre las ...,2025-03-17 07:30:00+01:00
1,MENORCA,https://www.menorca.info/deportes.html,NaN,2025-03-17 05:51:28+01:00
2,MENORCA,https://www.menorca.info/,NaN,2025-03-17 05:51:28+01:00
3,MENORCA,https://www.menorca.info/hemeroteca.html,NaN,2025-03-17 05:51:28+01:00
4,MENORCA,https://www.menorca.info/participa.html,NaN,2025-03-17 05:51:28+01:00
...,...,...,...,...
20768,Bilbao 24 Horas,https://www.xataka.com/robotica-e-ia/openai-se...,NaN,NaT
20769,Bilbao 24 Horas,https://www.xataka.com/robotica-e-ia/todos-mir...,NaN,NaT
20770,Bilbao 24 Horas,https://www.xataka.com/streaming/peliculas-ter...,NaN,NaT
20771,Bilbao 24 Horas,https://www.xataka.com/wearables/este-controla...,NaN,NaT


In [ ]:
data_df['fuente'].value_counts()

,count
fuente,
La Voz de Galicia,1645
La Vanguardia,1339
Gente Digital,1335
ABC,1210
MENORCA,751
...,...
Deportes Menorca,5
Diario del Puerto,5
Tomelloso,3
